# Model Training

Objective:
Build a machine learning pipeline capable of predicting SLA breaches at incident creation time.

The modeling phase includes:

- Preprocessing Pipeline
- Missing Value Handling
- Encoding
- Class Imbalance Handling
- Model Training
- Cross Validation
- Hyperparameter Tuning
- Model Selection

# Import Libraries

In [272]:
import pandas as pd
import numpy as np
import calendar
from pandas.api.types import is_object_dtype, is_string_dtype
from sklearn.utils.validation import check_is_fitted
from sklearn.base import BaseEstimator,TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder,TargetEncoder,FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score,recall_score,precision_score,f1_score,roc_auc_score
from sklearn.model_selection import cross_val_score



# Load Processed Data

In [273]:
X_train=pd.read_csv('../../data/processed/X_train.csv')
X_test=pd.read_csv('../../data/processed/X_test.csv')
y_train=pd.read_csv('../../data/processed/y_train.csv')
y_test=pd.read_csv('../../data/processed/y_test.csv')

# Feature Groups

| Feature Group      | Features                                                        |
| ------------------ | --------------------------------------------------------------- |
| Frequency Encoding | category, subcategory, assignment_group, assigned_to, opened_by |
| Target Encoding    | caller_id, location, u_symptom, contact_type                    |
| Ordinal Encoding   | impact, urgency, priority                                       |
| Numerical          | Hour                                                            |
| Calendar Features  | Month, Day_of_week                                              |
                                          |


## Preprocessing Pipeline Design

### Traditional ML Pipeline

Missing Value Replacement
        ↓
Ordinal Encoding
        ↓
Frequency Encoding
        ↓
Target Encoding
        ↓
Model

### CatBoost Pipeline

Missing Value Replacement
        ↓
Ordinal Encoding
        ↓
Native Categorical Handling
        ↓
CatBoost

# Custom Transformers

In [274]:

replacement={
    'location': 'Unknown_location',
    'category':'Unknown_category',
    'subcategory':'Unknown_subcategory',
    'u_symptom':'Unknown_symptom',
    'assignment_group':'Unknown_assignment_group',
    'assigned_to':'Unknown_assigned_to',
    'caller_id': 'Unknown_caller_id',
    'opened_by':'Unknown_opened_by'
}

nested_replacement={col: {'?':val} for col, val in replacement.items()}

def replace_missing_value(X):
    X_copy=X.copy()
    return X_copy.replace(nested_replacement)

missing_value_imputer = FunctionTransformer(replace_missing_value)

missing_values_pipeline=Pipeline([('impute_missing_value',missing_value_imputer)])



In [275]:
from sklearn.base import BaseEstimator, TransformerMixin

class BusinessMissingValueTransformer(BaseEstimator, TransformerMixin):

    def __init__(self):
        self.replacement = {
            'location': 'Unknown_location',
            'category': 'Unknown_category',
            'subcategory': 'Unknown_subcategory',
            'u_symptom': 'Unknown_symptom',
            'assignment_group': 'Unknown_assignment_group',
            'assigned_to': 'Unknown_assigned_to',
            'caller_id': 'Unknown_caller_id',
            'opened_by': 'Unknown_opened_by'
        }

        self.missing_tokens = ['?', 'NA', 'N/A', None]

    def fit(self, X, y=None):

        missing_cols = []

        for col in self.replacement.keys():
            if col not in X.columns:
                missing_cols.append(col)

        if missing_cols:
            raise ValueError(
                f"Required missing columns are: {missing_cols}"
            )

        return self

    def transform(self, X):

        X_copy = X.copy()

        for col, replacement_value in self.replacement.items():

            X_copy[col] = X_copy[col].replace(
                self.missing_tokens,
                replacement_value
            )

        return X_copy

In [276]:
missing_transformer = BusinessMissingValueTransformer()

missing_transformer.fit(X_train)

X_train_transformed = missing_transformer.transform(X_train)

In [277]:
(X_train_transformed[
    ['location','category','subcategory',
     'u_symptom','assignment_group',
     'assigned_to','caller_id','opened_by']
].isin(['?','NA','N/A']).sum())

location            0
category            0
subcategory         0
u_symptom           0
assignment_group    0
assigned_to         0
caller_id           0
opened_by           0
dtype: int64

In [278]:
X_train['priority'].value_counts()

priority
3 - Moderate    105915
4 - Low           3240
2 - High          2370
1 - Critical      1844
Name: count, dtype: int64

In [279]:
# Ordinal Transformer

class OrdinalTransformer(BaseEstimator,TransformerMixin):
    """
    Custom transformer to perform business-rule-based ordinal encoding.

    Features:
    - Validates required columns.
    - Detects unseen categories before transformation.
    - Encodes Impact, Urgency and Priority based on predefined business mappings.
    - Raises descriptive exceptions when unexpected categories are encountered.
    - Compatible with Scikit-Learn Pipelines.
    """
    def __init__(self,ordinal_mappings=None):
        if ordinal_mappings is None:
            self.ordinal_mappings={
                'impact':{
                    '3 - Low':1,
                    '2 - Medium':2,
                    '1 - High':3
                },
                'urgency':{
                    '3 - Low':1,
                    '2 - Medium':2,
                    '1 - High':3
                },
                'priority':{
                    '4 - Low':1,
                    '3 - Moderate':2,
                    '2 - High':3,
                    '1 - Critical':4
                }
            }
        else:
            """To build this as a reusable package, I am inject the mappings through the constructor with a default configuration."""
            self.ordinal_mappings = ordinal_mappings ## using this line we can use this transformer in any project

    def _validate_input(self,X):
         if not isinstance(X,pd.DataFrame):
              raise TypeError(f"Expected a pandas DataFrame, but got {type(X).__name__}")
    
    def _validate_columns(self,X):
        """Helper method to ensure all expected mapping columns exist in the dataset."""
        missing_columns=[col for col in self.ordinal_mappings.keys() if col not in X.columns]
        if missing_columns:
            raise ValueError(f"Required missing columns are :{missing_columns}") 
        

    def fit(self,X,y=None):
        """
        Validate that all required ordinal columns are present.
        """
        self._validate_input(X)
        self._validate_columns(X)
        return self

    def transform(self,X):
        """
        Validate ordinal values and apply business-rule-based encoding.
        """
        self._validate_input(X)
        self._validate_columns(X)
        
        X_copy=X.copy()
        invalid_values={}

        for col,mapping in self.ordinal_mappings.items():
            diff=set(X_copy[col].unique()) - set(mapping.keys())
            if diff:
                 invalid_values[col]=diff
        
        if invalid_values:
            raise ValueError(f"Unknown categorical values found during transformation: {invalid_values}")
        
        for col,value in self.ordinal_mappings.items():
            X_copy[col]=X_copy[col].map(value)
            
        return X_copy
        
ordinal_transformer = OrdinalTransformer()

X_train_ordinal = ordinal_transformer.fit_transform(X_train)
X_train_ordinal[['impact','urgency','priority']]

,impact,urgency,priority
0,2,2,2
1,2,2,2
2,2,2,2
3,2,2,2
4,2,2,2
...,...,...,...
113364,2,2,2
113365,2,2,2
113366,2,2,2
113367,2,2,2


In [280]:
#Frequency Encoder

class FrequencyEncoder(BaseEstimator,TransformerMixin):
    """
    Custom transformer to perform frequency encoding.

    Features:
        - Learns normalized frequencies from training data.
        - Encodes configured high-cardinality categorical features.
        - Handles unseen categories using a configurable unknown_value.
        - Compatible with Scikit-Learn Pipelines.
    """
    def __init__(self,frequency_columns=None,unknown_value=0.0):
        if frequency_columns is None:
            self.frequency_columns=[
            'category',
            'subcategory',
            'assignment_group',
            'assigned_to',
            'opened_by'
            ]
        else:
            self.frequency_columns=frequency_columns

        self.frequency_mappings={}
        self.unknown_value = unknown_value
    
    def _validate_input(self,X):
        if not isinstance(X,pd.DataFrame):
              raise TypeError(f"Expected a pandas DataFrame, but got {type(X).__name__}")
    
    def _validate_columns(self,X):
        missing_columns=[col for col in self.frequency_columns if col not in X.columns]
        if missing_columns:
            raise ValueError(f"Required missing columns are :{missing_columns}")
        
    def fit(self,X,y=None):
        self._validate_input(X)
        self._validate_columns(X)
        self.frequency_mappings={
          col:X[col].value_counts(normalize=True).to_dict()  for col in self.frequency_columns
        }
        
        return self

    def transform(self,X,y=None):
        self._validate_input(X)
        self._validate_columns(X)
        X_copy=X.copy()

        if not self.frequency_mappings:
            raise ValueError(
                "FrequencyEncoder has not been fitted yet. "
                "Call fit() before transform()."
                )
        
        for col, mapping in self.frequency_mappings.items():
            X_copy[col] = X_copy[col].map(mapping)

            X_copy[col] = X_copy[col].fillna(self.unknown_value)
            
        return X_copy


encoder = FrequencyEncoder()

encoder.fit(X_train)

X_train_encoded = encoder.transform(X_train)
X_train_encoded


,contact_type,location,category,subcategory,u_symptom,assignment_group,assigned_to,caller_id,opened_by,impact,urgency,priority,Hour,Day_of_week,Month
0,Phone,Location 51,0.113276,0.110004,Symptom 534,0.306724,0.194304,Caller 1338,0.291623,2 - Medium,2 - Medium,3 - Moderate,15,Friday,May
1,Phone,Location 143,0.130538,0.055438,Symptom 116,0.025536,0.014201,Caller 1638,0.021055,2 - Medium,2 - Medium,3 - Moderate,21,Thursday,March
2,Phone,Location 143,0.045833,0.024204,Symptom 491,0.004675,0.005090,Caller 3933,0.057132,2 - Medium,2 - Medium,3 - Moderate,8,Monday,May
3,Phone,Location 143,0.036421,0.005363,Symptom 491,0.025536,0.012949,Caller 3423,0.043336,2 - Medium,2 - Medium,3 - Moderate,16,Tuesday,May
4,Phone,Location 143,0.051028,0.028897,Symptom 105,0.306724,0.194304,Caller 3325,0.291623,2 - Medium,2 - Medium,3 - Moderate,20,Tuesday,April
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
113364,Phone,Location 143,0.039164,0.013840,Symptom 517,0.306724,0.074923,Caller 1250,0.057132,2 - Medium,2 - Medium,3 - Moderate,9,Friday,May
113365,Phone,Location 38,0.093862,0.007736,?,0.010964,0.006757,Caller 2847,0.025377,2 - Medium,2 - Medium,3 - Moderate,10,Wednesday,March
113366,Phone,Location 204,0.130538,0.074368,?,0.100733,0.074923,Caller 2852,0.026586,2 - Medium,2 - Medium,3 - Moderate,12,Monday,March
113367,Phone,Location 143,0.051972,0.252820,?,0.025536,0.014201,Caller 3416,0.043336,2 - Medium,2 - Medium,3 - Moderate,7,Sunday,May


In [281]:
encoder = FrequencyEncoder(unknown_value=0.0)
encoder.fit(X_train)

# Clean naming convention to keep track of splits
X_test_encoded = encoder.transform(X_test)
X_test_encoded

,contact_type,location,category,subcategory,u_symptom,assignment_group,assigned_to,caller_id,opened_by,impact,urgency,priority,Hour,Day_of_week,Month
0,Phone,Location 188,0.093862,0.252820,Symptom 152,0.017121,0.001076,Caller 4036,0.020870,2 - Medium,2 - Medium,3 - Moderate,1,Sunday,April
1,Phone,Location 204,0.028103,0.252820,?,0.053833,0.007798,Caller 5113,0.029364,2 - Medium,2 - Medium,3 - Moderate,14,Tuesday,May
2,Phone,Location 37,0.093862,0.007736,Symptom 218,0.010964,0.004578,Caller 3062,0.013716,2 - Medium,2 - Medium,3 - Moderate,16,Wednesday,March
3,Phone,Location 56,0.046159,0.023710,Symptom 244,0.020120,0.002840,Caller 4733,0.291623,2 - Medium,2 - Medium,3 - Moderate,8,Monday,April
4,Phone,Location 204,0.051028,0.028897,Symptom 105,0.306724,0.074923,Caller 129,0.057132,2 - Medium,2 - Medium,3 - Moderate,8,Tuesday,April
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28338,Phone,Location 204,0.055456,0.252820,Symptom 491,0.100733,0.074923,Caller 944,0.020870,2 - Medium,2 - Medium,3 - Moderate,16,Monday,March
28339,Phone,Location 143,0.113276,0.110004,Symptom 534,0.100733,0.074923,Caller 2498,0.025377,2 - Medium,2 - Medium,3 - Moderate,9,Monday,March
28340,Phone,Location 108,0.130538,0.055438,Symptom 295,0.100733,0.194304,Caller 5436,0.051222,2 - Medium,2 - Medium,3 - Moderate,8,Friday,May
28341,Phone,Location 51,0.051972,0.055438,?,0.043883,0.003211,Caller 4934,0.033678,2 - Medium,2 - Medium,3 - Moderate,9,Friday,May


In [282]:
# Filters the DataFrame to only your processed columns and checks for 0.0
zero_counts = (X_test_encoded[encoder.frequency_columns] == 0.0).sum()

print("Count of 0.0 values per column:")
print(zero_counts)


Count of 0.0 values per column:
category            0
subcategory         2
assignment_group    0
assigned_to         3
opened_by           3
dtype: int64


In [283]:
class TargetEncoder(BaseEstimator, TransformerMixin):
    """Custom transformer to perform target encoding with additive smoothing.

    This encoder replaces categorical values with the smoothed mean of the 
    target variable for that specific category. It balances the local category 
    mean against the global target mean to mitigate overfitting in high-cardinality 
    features or small sample sizes.

    Parameters
    ----------
    target_columns : list of str, default=None
        The columns in the input DataFrame to be target encoded. If None, 
        defaults to ['caller_id', 'location', 'u_symptom', 'contact_type'].
    target_column : str, default='made_sla'
        The column name of the target variable used to search inside `y` if 
        it is passed as a pandas DataFrame.
    smoothing : int or float, default=10
        The smoothing factor (m-estimate constant). Higher values pull category 
        means closer to the global mean, acting as a stronger regularizer.

    Attributes
    ----------
    target_mappings : dict
        Nested dictionary mapping feature names to dictionaries of encoded category values.
    global_mean : float
        The overall mean of the target variable across the entire training set.
    n_features_in_ : tuple
        The shape dimensions of the feature matrix seen during `fit`.
    """
    
    def __init__(self, target_columns=None, target_column='made_sla', smoothing=10):
        if target_columns is None:
            self.target_columns = ['caller_id', 'location', 'u_symptom', 'contact_type']
        else:
            self.target_columns = target_columns
            
        self.target_column = target_column
        self.smoothing = smoothing
        self.target_mappings = {}
        self.global_mean = None

    def _validate_X(self, X):
        """Validates input structural data type for the feature matrix X."""
        if not isinstance(X, pd.DataFrame):
            raise TypeError(f"Expected X to be a pandas DataFrame, but got {type(X).__name__}")

    def _validate_y(self, y):
        """Validates input structural data types, dimensions, and naming for the target vector y."""
        if not isinstance(y, (pd.Series, pd.DataFrame, np.ndarray)):
            raise TypeError(f"Expected y to be a Series, DataFrame, or array, but got {type(y).__name__}")
        
        if isinstance(y, pd.DataFrame) and self.target_column not in y.columns:
            raise ValueError(f"The target column '{self.target_column}' is missing from the target DataFrame.")
            
        if isinstance(y, np.ndarray) and y.ndim != 1:
            raise ValueError(f"Expected y to be a 1D array, but got a {y.ndim}D array instead.")

    def _validate_columns(self, X):
        """Verifies that all specified feature columns exist in the DataFrame."""
        missing_columns = [col for col in self.target_columns if col not in X.columns]
        if missing_columns:    
            raise ValueError(f"Required missing columns are: {missing_columns}")

    def _compute_smoothed_mean(self, group_stats):
        """Calculates the smoothed target value using the explicit local variables layout."""
        numerator = (group_stats['count'] * group_stats['mean']) + (self.smoothing * self.global_mean)
        denominator = group_stats['count'] + self.smoothing
        return numerator / denominator

    def fit(self, X, y):
        """Fit the target encoder by calculating smoothed category means.

        Parameters
        ----------
        X : pandas.DataFrame
            The feature matrix containing columns to encode.
        y : pandas.Series, pandas.DataFrame, or numpy.ndarray
            The target vector.

        Returns
        -------
        self : object
            Returns the instance itself.
        """
        self._validate_X(X)
        self._validate_y(y)
        self._validate_columns(X)
        
        self.n_features_in_ = X.shape[1]
        
        if isinstance(y, pd.DataFrame):
            y_series = y[self.target_column]
        elif isinstance(y, pd.Series):
            y_series = y
        else:
            y_series = pd.Series(y, index=X.index)

        self.global_mean = y_series.mean()
        self.target_mappings = {}

        for col in self.target_columns:
            encoding_df = pd.concat([X[col], y_series.rename(self.target_column)], axis=1)
            group_stats = encoding_df.groupby(col)[self.target_column].agg(['mean', 'count'])

            group_stats['smoothed'] = self._compute_smoothed_mean(group_stats)
            self.target_mappings[col] = group_stats['smoothed'].to_dict()
             
        return self
    
    def transform(self, X):
        """Transform the categorical columns using the learned target mappings.

        Unseen categories encountered during testing will be imputed with 
        the global training mean.

        Parameters
        ----------
        X : pandas.DataFrame
            The feature matrix to transform.

        Returns
        -------
        X_transformed : pandas.DataFrame
            The transformed DataFrame with numerical target-encoded features.
        """
        self._validate_X(X)
        self._validate_columns(X)
        
        check_is_fitted(self, "n_features_in_")

        X_transformed = X.copy()
        for col, mapping in self.target_mappings.items():
            X_transformed[col] = (X_transformed[col].map(mapping).fillna(self.global_mean))
    

        return X_transformed
                 

    def get_feature_names_out(self, input_features=None):
        """Returns feature names out for Scikit-Learn pipeline support."""
        if input_features is None:
            return np.array(self.target_columns)
        return np.array(input_features)

In [284]:
target_encoder=TargetEncoder().fit(X_train,y_train)

In [285]:
target_encoder.transform(X_test)

,contact_type,location,category,subcategory,u_symptom,assignment_group,assigned_to,caller_id,opened_by,impact,urgency,priority,Hour,Day_of_week,Month
0,0.935146,0.988166,Category 46,Subcategory 174,0.952850,Group 55,Resolver 45,0.923411,Opened by 390,2 - Medium,2 - Medium,3 - Moderate,1,Sunday,April
1,0.935146,0.927769,Category 34,Subcategory 174,0.922440,Group 25,Resolver 227,0.913497,Opened by 501,2 - Medium,2 - Medium,3 - Moderate,14,Tuesday,May
2,0.935146,0.911855,Category 46,Subcategory 251,0.900964,Group 22,Resolver 133,0.911732,Opened by 500,2 - Medium,2 - Medium,3 - Moderate,16,Wednesday,March
3,0.935146,0.927777,Category 57,Subcategory 170,0.938651,Group 65,Resolver 25,0.981937,Opened by 17,2 - Medium,2 - Medium,3 - Moderate,8,Monday,April
4,0.935146,0.927769,Category 32,Subcategory 9,0.968336,Group 70,Resolver 17,0.974990,Opened by 24,2 - Medium,2 - Medium,3 - Moderate,8,Tuesday,April
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28338,0.935146,0.927769,Category 23,Subcategory 174,0.939732,?,Resolver 17,0.914686,Opened by 390,2 - Medium,2 - Medium,3 - Moderate,16,Monday,March
28339,0.935146,0.930394,Category 42,Subcategory 223,0.985770,?,Resolver 17,0.931239,Opened by 397,2 - Medium,2 - Medium,3 - Moderate,9,Monday,March
28340,0.935146,0.942683,Category 26,Subcategory 164,0.991870,?,?,0.957685,Opened by 131,2 - Medium,2 - Medium,3 - Moderate,8,Friday,May
28341,0.935146,0.933127,Category 9,Subcategory 164,0.922440,Group 20,Resolver 184,0.910313,Opened by 62,2 - Medium,2 - Medium,3 - Moderate,9,Friday,May


In [286]:
X_train[['Hour', 'Month', 'Day_of_week']].dtypes

Hour           int64
Month            str
Day_of_week      str
dtype: object

In [287]:
X_train[['Hour', 'Month', 'Day_of_week']].describe()

,Hour
count,113369.000000
mean,11.887579
std,4.016466
min,0.000000
25%,9.000000
50%,11.000000
75%,15.000000
max,23.000000


In [288]:
X_train[ 'Month'].unique()

<StringArray>
[      'May',     'March',     'April',  'December',  'February',   'October',
    'August',   'January',  'November',      'July', 'September',      'June']
Length: 12, dtype: str

In [289]:
class CyclicEncoder(BaseEstimator, TransformerMixin):
    """
    Parameters
    ----------
        cyclic_columns : dict, default=None
            Dictionary mapping cyclic feature names to their cycle lengths.

        drop_original : bool, default=True
            Whether to remove the original cyclic columns after encoding.

    Attributes
    ----------
        month_mapping : dict
        day_mapping : dict
        n_features_in_ : int
    """
    def __init__(self,cyclic_columns=None,drop_original=True):
        if cyclic_columns is None:
            self.cyclic_columns={
                'Hour':24,
                'Day_of_week':7,
                'Month':12
            }
        else:
            self.cyclic_columns=cyclic_columns

        self.drop_original=drop_original
        self.month_mapping={
            month:idx for idx,month in enumerate(calendar.month_name) if month
            }
        self.day_mapping={ 
            day:idx for idx,day in enumerate(calendar.day_name) if day
            }
        
        self.category_mappings = {
            "Month": self.month_mapping,
            "Day_of_week": self.day_mapping
        }
        
    
    def _validate_input(self,X):
        if not isinstance(X,pd.DataFrame):
            raise TypeError(f"Expected the pandas Dataframe to be, but got: {type(X).__name__} ")
    
    def _validate_columns(self,X):
        missing_columns=[col for col in self.cyclic_columns if col not in X.columns]
        if missing_columns:    
            raise ValueError(f"Required missing columns are: {missing_columns}") 

    def fit(self,X,y=None):
        self._validate_input(X)
        self._validate_columns(X)
        self.n_features_in_ = X.shape[1]
        return self

    def transform(self,X):
        self._validate_input(X)
        self._validate_columns(X)
        check_is_fitted(self, "n_features_in_")
        X_transformed=X.copy()

        for col, mapping in self.category_mappings.items():
            if col in self.cyclic_columns:
                if is_object_dtype(X_transformed[col]) or is_string_dtype(X_transformed[col]):
                    X_transformed[col] = X_transformed[col].map(mapping)

        for col,period in self.cyclic_columns.items():
                        
            numeric_values = pd.to_numeric(X_transformed[col],errors='raise')

            X_transformed[f'{col}_sin']=np.sin((2*np.pi*numeric_values)/period)
            X_transformed[f'{col}_cos']=np.cos((2*np.pi*numeric_values)/period)
            
        if self.drop_original:
            X_transformed=X_transformed.drop(columns=list(self.cyclic_columns.keys()),errors='ignore')

        return X_transformed
        
        
        

In [290]:
cyclic=CyclicEncoder().fit(X_train,y_train)

In [291]:
cyclic.transform(X_test)

,contact_type,location,category,subcategory,u_symptom,assignment_group,assigned_to,caller_id,opened_by,impact,urgency,priority,Hour_sin,Hour_cos,Day_of_week_sin,Day_of_week_cos,Month_sin,Month_cos
0,Phone,Location 188,Category 46,Subcategory 174,Symptom 152,Group 55,Resolver 45,Caller 4036,Opened by 390,2 - Medium,2 - Medium,3 - Moderate,0.258819,0.965926,-0.781831,0.623490,0.866025,-5.000000e-01
1,Phone,Location 204,Category 34,Subcategory 174,?,Group 25,Resolver 227,Caller 5113,Opened by 501,2 - Medium,2 - Medium,3 - Moderate,-0.500000,-0.866025,0.781831,0.623490,0.500000,-8.660254e-01
2,Phone,Location 37,Category 46,Subcategory 251,Symptom 218,Group 22,Resolver 133,Caller 3062,Opened by 500,2 - Medium,2 - Medium,3 - Moderate,-0.866025,-0.500000,0.974928,-0.222521,1.000000,6.123234e-17
3,Phone,Location 56,Category 57,Subcategory 170,Symptom 244,Group 65,Resolver 25,Caller 4733,Opened by 17,2 - Medium,2 - Medium,3 - Moderate,0.866025,-0.500000,0.000000,1.000000,0.866025,-5.000000e-01
4,Phone,Location 204,Category 32,Subcategory 9,Symptom 105,Group 70,Resolver 17,Caller 129,Opened by 24,2 - Medium,2 - Medium,3 - Moderate,0.866025,-0.500000,0.781831,0.623490,0.866025,-5.000000e-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28338,Phone,Location 204,Category 23,Subcategory 174,Symptom 491,?,Resolver 17,Caller 944,Opened by 390,2 - Medium,2 - Medium,3 - Moderate,-0.866025,-0.500000,0.000000,1.000000,1.000000,6.123234e-17
28339,Phone,Location 143,Category 42,Subcategory 223,Symptom 534,?,Resolver 17,Caller 2498,Opened by 397,2 - Medium,2 - Medium,3 - Moderate,0.707107,-0.707107,0.000000,1.000000,1.000000,6.123234e-17
28340,Phone,Location 108,Category 26,Subcategory 164,Symptom 295,?,?,Caller 5436,Opened by 131,2 - Medium,2 - Medium,3 - Moderate,0.866025,-0.500000,-0.433884,-0.900969,0.500000,-8.660254e-01
28341,Phone,Location 51,Category 9,Subcategory 164,?,Group 20,Resolver 184,Caller 4934,Opened by 62,2 - Medium,2 - Medium,3 - Moderate,0.707107,-0.707107,-0.433884,-0.900969,0.500000,-8.660254e-01


In [292]:
month_mapping={
            month:idx for idx,month in enumerate(calendar.month_name) if month
            }
for col,mapping in month_mapping.items():
    print(col,mapping)

January 1
February 2
March 3
April 4
May 5
June 6
July 7
August 8
September 9
October 10
November 11
December 12


## Modeling Strategy

Baseline Models
- Logistic Regression
- Decision Tree
- Random Forest

Boosting Models
- XGBoost
- LightGBM
- CatBoost

Evaluation Metrics
- Precision
- Recall
- F1 Score
- ROC-AUC
- PR-AUC

Cross Validation
- Stratified K-Fold

Hyperparameter Optimization
- Optuna

# Preprocessing Pipeline

In [293]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (accuracy_score,
                             precision_score,
                             recall_score,
                             f1_score,
                             roc_auc_score,
                             confusion_matrix,
                             classification_report
                        )                   


In [294]:
preprocessing_pipeline=Pipeline(steps=[
    ("Business_missing",BusinessMissingValueTransformer()),
    ("ordinal_encoding",OrdinalTransformer()),
    ("Frequency_encoding",FrequencyEncoder()),
    ("Target_encoding",TargetEncoder()),
    ("Cyclic_encoding",CyclicEncoder())
])

X_train_processed=preprocessing_pipeline.fit_transform(X_train,y_train)
X_test_processed=preprocessing_pipeline.transform(X_test)

## Preprocessing Pipeline Validation

In [295]:
X_train_processed.shape,X_test_processed.shape

((113369, 18), (28343, 18))

In [296]:
X_train_processed.dtypes,X_test_processed.dtypes

(contact_type        float64
 location            float64
 category            float64
 subcategory         float64
 u_symptom           float64
 assignment_group    float64
 assigned_to         float64
 caller_id           float64
 opened_by           float64
 impact                int64
 urgency               int64
 priority              int64
 Hour_sin            float64
 Hour_cos            float64
 Day_of_week_sin     float64
 Day_of_week_cos     float64
 Month_sin           float64
 Month_cos           float64
 dtype: object,
 contact_type        float64
 location            float64
 category            float64
 subcategory         float64
 u_symptom           float64
 assignment_group    float64
 assigned_to         float64
 caller_id           float64
 opened_by           float64
 impact                int64
 urgency               int64
 priority              int64
 Hour_sin            float64
 Hour_cos            float64
 Day_of_week_sin     float64
 Day_of_week_cos     float6

In [297]:
X_train_processed.isnull().sum(),X_test_processed.isnull().sum()

(contact_type        0
 location            0
 category            0
 subcategory         0
 u_symptom           0
 assignment_group    0
 assigned_to         0
 caller_id           0
 opened_by           0
 impact              0
 urgency             0
 priority            0
 Hour_sin            0
 Hour_cos            0
 Day_of_week_sin     0
 Day_of_week_cos     0
 Month_sin           0
 Month_cos           0
 dtype: int64,
 contact_type        0
 location            0
 category            0
 subcategory         0
 u_symptom           0
 assignment_group    0
 assigned_to         0
 caller_id           0
 opened_by           0
 impact              0
 urgency             0
 priority            0
 Hour_sin            0
 Hour_cos            0
 Day_of_week_sin     0
 Day_of_week_cos     0
 Month_sin           0
 Month_cos           0
 dtype: int64)

In [298]:
X_train_processed.columns,X_test_processed.columns

(Index(['contact_type', 'location', 'category', 'subcategory', 'u_symptom',
        'assignment_group', 'assigned_to', 'caller_id', 'opened_by', 'impact',
        'urgency', 'priority', 'Hour_sin', 'Hour_cos', 'Day_of_week_sin',
        'Day_of_week_cos', 'Month_sin', 'Month_cos'],
       dtype='str'),
 Index(['contact_type', 'location', 'category', 'subcategory', 'u_symptom',
        'assignment_group', 'assigned_to', 'caller_id', 'opened_by', 'impact',
        'urgency', 'priority', 'Hour_sin', 'Hour_cos', 'Day_of_week_sin',
        'Day_of_week_cos', 'Month_sin', 'Month_cos'],
       dtype='str'))

In [299]:
X_train_processed.columns==X_test_processed.columns

array([ True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True])

In [300]:
X_train_processed.describe()

,contact_type,location,category,subcategory,u_symptom,assignment_group,assigned_to,caller_id,opened_by,impact,urgency,priority,Hour_sin,Hour_cos,Day_of_week_sin,Day_of_week_cos,Month_sin,Month_cos
count,113369.000000,113369.000000,113369.000000,113369.000000,113369.000000,113369.000000,113369.000000,113369.000000,113369.000000,113369.000000,113369.000000,113369.000000,113369.000000,113369.000000,113369.000000,113369.000000,113369.000000,1.133690e+05
mean,0.934976,0.935025,0.071071,0.090099,0.935183,0.118600,0.053411,0.933935,0.105085,1.997398,2.003167,2.024857,0.023445,-0.593585,0.318834,0.039202,0.810530,-3.748158e-01
std,0.003999,0.014044,0.041438,0.100276,0.021079,0.127754,0.072309,0.030795,0.120478,0.229214,0.232606,0.337534,0.653210,0.469499,0.551591,0.769781,0.231835,3.857572e-01
min,0.835214,0.821144,0.000009,0.000009,0.790731,0.000009,0.000009,0.679620,0.000009,1.000000,1.000000,1.000000,-1.000000,-1.000000,-0.974928,-0.900969,-1.000000,-1.000000e+00
25%,0.935146,0.927769,0.036421,0.008018,0.922440,0.015489,0.006007,0.915665,0.023366,2.000000,2.000000,2.000000,-0.707107,-0.866025,0.000000,-0.900969,0.500000,-8.660254e-01
50%,0.935146,0.933127,0.055456,0.028897,0.939732,0.047923,0.013540,0.936528,0.034445,2.000000,2.000000,2.000000,0.258819,-0.707107,0.433884,-0.222521,0.866025,-5.000000e-01
75%,0.935146,0.942683,0.112165,0.252820,0.939732,0.306724,0.074923,0.956572,0.291623,2.000000,2.000000,2.000000,0.707107,-0.500000,0.781831,0.623490,1.000000,6.123234e-17
max,0.975916,0.988166,0.130538,0.252820,0.992165,0.306724,0.194304,0.994444,0.291623,3.000000,3.000000,4.000000,1.000000,1.000000,0.974928,1.000000,1.000000,1.000000e+00


In [301]:
X_test_processed.describe()


,contact_type,location,category,subcategory,u_symptom,assignment_group,assigned_to,caller_id,opened_by,impact,urgency,priority,Hour_sin,Hour_cos,Day_of_week_sin,Day_of_week_cos,Month_sin,Month_cos
count,28343.000000,28343.000000,28343.000000,28343.000000,28343.000000,28343.000000,28343.000000,28343.000000,28343.000000,28343.000000,28343.000000,28343.000000,28343.000000,28343.000000,28343.000000,28343.000000,28343.000000,2.834300e+04
mean,0.934969,0.934997,0.070970,0.089695,0.935132,0.118418,0.053059,0.933718,0.106536,1.996472,2.002223,2.022581,0.026444,-0.597546,0.317181,0.043982,0.810115,-3.761496e-01
std,0.003973,0.013895,0.041348,0.100152,0.021163,0.127931,0.072166,0.030757,0.120945,0.223808,0.228736,0.327160,0.655625,0.460888,0.552573,0.769517,0.231781,3.853686e-01
min,0.835214,0.821144,0.000009,0.000000,0.790731,0.000009,0.000000,0.679620,0.000000,1.000000,1.000000,1.000000,-1.000000,-1.000000,-0.974928,-0.900969,-1.000000,-1.000000e+00
25%,0.935146,0.927769,0.036421,0.007736,0.922440,0.015489,0.006007,0.915450,0.023366,2.000000,2.000000,2.000000,-0.707107,-0.866025,0.000000,-0.900969,0.500000,-8.660254e-01
50%,0.935146,0.933127,0.055456,0.028897,0.939732,0.047923,0.013540,0.936528,0.034445,2.000000,2.000000,2.000000,0.258819,-0.707107,0.433884,-0.222521,0.866025,-5.000000e-01
75%,0.935146,0.942683,0.112165,0.252820,0.939732,0.306724,0.074923,0.955829,0.291623,2.000000,2.000000,2.000000,0.707107,-0.500000,0.781831,0.623490,1.000000,6.123234e-17
max,0.975916,0.988166,0.130538,0.252820,0.992165,0.306724,0.194304,0.994444,0.291623,3.000000,3.000000,4.000000,1.000000,1.000000,0.974928,1.000000,1.000000,1.000000e+00


In [302]:
print(f"Training Shape : {X_train_processed.shape}")
print(f"Testing Shape  : {X_test_processed.shape}")

print("\nTraining Missing Values")
print(X_train_processed.isna().sum().sum())

print("\nTesting Missing Values")
print(X_test_processed.isna().sum().sum())

print("\nChecking '?' values in Training Data")
print((X_train_processed == '?').sum())

print("\nChecking '?' values in Testing Data")
print((X_test_processed == '?').sum())

Training Shape : (113369, 18)
Testing Shape  : (28343, 18)

Training Missing Values
0

Testing Missing Values
0

Checking '?' values in Training Data
contact_type        0
location            0
category            0
subcategory         0
u_symptom           0
assignment_group    0
assigned_to         0
caller_id           0
opened_by           0
impact              0
urgency             0
priority            0
Hour_sin            0
Hour_cos            0
Day_of_week_sin     0
Day_of_week_cos     0
Month_sin           0
Month_cos           0
dtype: int64

Checking '?' values in Testing Data
contact_type        0
location            0
category            0
subcategory         0
u_symptom           0
assignment_group    0
assigned_to         0
caller_id           0
opened_by           0
impact              0
urgency             0
priority            0
Hour_sin            0
Hour_cos            0
Day_of_week_sin     0
Day_of_week_cos     0
Month_sin           0
Month_cos           0
dtype: i

In [303]:
logestic_regression=LogisticRegression(class_weight='balanced',random_state=42,max_iter=1000)

In [304]:
y_train_processed=y_train.to_numpy().reshape(-1,)

In [305]:
logestic_regression.fit(X_train_processed,y_train.values.ravel())

,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",42
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default 

In [306]:
y_pred=logestic_regression.predict(X_test_processed)

In [307]:
y_prob = logestic_regression.predict_proba(X_test_processed)[:, 1]

In [308]:
baseline_results = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred),
    "Recall": recall_score(y_test, y_pred),
    "F1 Score": f1_score(y_test, y_pred),
    "ROC-AUC": roc_auc_score(y_test, y_prob)
}

for metric, value in baseline_results.items():
    print(f"{metric:<12}: {value:.4f}")

Accuracy    : 0.6564
Precision   : 0.9502
Recall      : 0.6675
F1 Score    : 0.7841
ROC-AUC     : 0.6423


In [309]:
def evaluate_model(model_name,y_true,y_pred,y_prob):
    """
    Evaluate classification model performance.

    Parameters
    ----------
    y_true : array-like
        Actual target values.

    y_pred : array-like
        Predicted class labels.

    y_prob : array-like
        Predicted probabilities for the positive class.

    Returns
    -------
    dict
        Dictionary containing evaluation metrics and reports.
    """
    report_df = pd.DataFrame(classification_report(y_true, y_pred, output_dict=True)).T
    logistic_results={
        "Model":model_name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "F1 Score": f1_score(y_true, y_pred),
        "ROC-AUC": roc_auc_score(y_true, y_prob),
        "Test Samples": len(y_true),
        "Confusion Matrix": confusion_matrix(y_true, y_pred),
        "Classification Report": report_df
    }
    for metric, value in logistic_results.items():
        if metric not in ["Confusion Matrix", "Classification Report"]:
            print(f"{metric:<15}: {value}")
            
    print("\nConfusion Matrix")
    print(logistic_results["Confusion Matrix"])

    print("\nClassification Report")
    display(logistic_results["Classification Report"])


In [310]:
evaluate_model('Logistic Regression',y_true=y_test,y_pred=y_pred,y_prob=y_prob)

Model          : Logistic Regression
Accuracy       : 0.6563878206259041
Precision      : 0.9502014504431909
Recall         : 0.6674716981132075
F1 Score       : 0.7841294469688573
ROC-AUC        : 0.6422709384821712
Test Samples   : 28343

Confusion Matrix
[[  916   927]
 [ 8812 17688]]

Classification Report


,precision,recall,f1-score,support
False,0.094161,0.497016,0.158327,1843.000000
True,0.950201,0.667472,0.784129,26500.000000
accuracy,0.656388,0.656388,0.656388,0.656388
macro avg,0.522181,0.582244,0.471228,28343.000000
weighted avg,0.894538,0.656388,0.743437,28343.000000


In [311]:
decision_tree=DecisionTreeClassifier(criterion='gini',max_depth=10,class_weight='balanced',random_state=42)

In [312]:
decision_tree.fit(X_train_processed,y_train.values.ravel())

,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",10
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary <random_state>` for details.",42
,"class_weight class_weight: dict, list of dict or ""balanced"", default=NoneWeights associated with classes in the form ``{class_label: weight}``.If None, all classes are supposed to have weight one. Formulti-output problems, a list of dicts can be provided in the sameorder as the columns of y.Note that for multioutput (including multilabel) weights should bedefined for each class of every column in its own dict. For example,for four-class multilabel classification weights should be[{0: 1, 1: 1}, {0: 1, 1: 5}, {0: 1, 1: 1}, {0: 1, 1: 1}] instead of[{1:1}, {2:5}, {3:1}, {4:1}].The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``For multi-output, the weights of each column of y will be multiplied.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified.",'balanced'
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are co

In [313]:
y_pred_dt=decision_tree.predict(X_test_processed)

In [314]:
y_prob_dt=decision_tree.predict_proba(X_test_processed)[:,1]

In [315]:
evaluate_model('Decision Tree',y_true=y_test,y_pred=y_pred_dt,y_prob=y_prob_dt)

Model          : Decision Tree
Accuracy       : 0.5987721836079455
Precision      : 0.9567081270377974
Recall         : 0.5979245283018868
F1 Score       : 0.7359156564952859
ROC-AUC        : 0.626149274664974
Test Samples   : 28343

Confusion Matrix
[[ 1126   717]
 [10655 15845]]

Classification Report


,precision,recall,f1-score,support
False,0.095578,0.610960,0.165297,1843.000000
True,0.956708,0.597925,0.735916,26500.000000
accuracy,0.598772,0.598772,0.598772,0.598772
macro avg,0.526143,0.604442,0.450606,28343.000000
weighted avg,0.900713,0.598772,0.698811,28343.000000


In [316]:
random_forest=RandomForestClassifier(n_estimators=100,class_weight='balanced',max_depth=10,max_features='sqrt',bootstrap=True,criterion='gini',random_state=42)

In [317]:
random_forest.fit(X_train_processed,y_train.values.ravel())

,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",10
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"class_weight class_weight: {""balanced"", ""balanced_subsample""}, dict or list of dicts, default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one. Formulti-output problems, a list of dicts can be provided in the sameorder as the columns of y.Note that for multioutput (including multilabel) weights should bedefined for each class of every column in its own dict. For example,for four-class multilabel classification weights should be[{0: 1, 1: 1}, {0: 1, 1: 5}, {0: 1, 1: 1}, {0: 1, 1: 1}] instead of[{1:1}, {2:5}, {3:1}, {4:1}].The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``The ""balanced_subsample"" mode is the same as ""balanced"" except thatweights are computed based on the bootstrap sample for every treegrown.For multi-output, the weights of each column of y will be multiplied.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified.",'balanced'
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 

In [318]:
y_pred_rf=random_forest.predict(X_test_processed)


In [319]:
y_prob_rf=random_forest.predict_proba(X_test_processed)[:,1]


In [320]:
evaluate_model('Random Forest',y_true=y_test.values.ravel(),y_pred=y_pred_rf,y_prob=y_prob_rf)

Model          : Random Forest
Accuracy       : 0.6309494407790284
Precision      : 0.954339451620213
Recall         : 0.6356981132075472
F1 Score       : 0.7630911396992208
ROC-AUC        : 0.6473550200145374
Test Samples   : 28343

Confusion Matrix
[[ 1037   806]
 [ 9654 16846]]

Classification Report


,precision,recall,f1-score,support
False,0.096997,0.562670,0.165470,1843.000000
True,0.954339,0.635698,0.763091,26500.000000
accuracy,0.630949,0.630949,0.630949,0.630949
macro avg,0.525668,0.599184,0.464281,28343.000000
weighted avg,0.898591,0.630949,0.724231,28343.000000


# Hyperparamters Tuning

In [321]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
import optuna
from sklearn.model_selection import StratifiedKFold

In [322]:
def objective(trial):
    n_estimators=trial.suggest_int('n_estimators',50,500,step=50)
    max_depth=trial.suggest_int('max_depth',5,19,step=2)
    min_samples_split=trial.suggest_int('min_samples_split',2,20,step=1)
    min_samples_leaf=trial.suggest_int('min_samples_leaf',1,10,step=1)
    max_features=trial.suggest_categorical('max_features',['sqrt','log2',None])
    
    random_forest_tune=RandomForestClassifier(n_estimators=n_estimators,max_depth=max_depth,
                                              min_samples_split=min_samples_split,min_samples_leaf=min_samples_leaf,
                                              max_features=max_features,class_weight='balanced',
                                              bootstrap=True,criterion='gini',random_state=42,n_jobs=-1)
    
    cv_f1_score=[]
    skf=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)

    for train_idx,valid_idx in skf.split(X_train_processed,y_train):
        X_fold_train=X_train_processed.iloc[train_idx]
        X_fold_valid=X_train_processed.iloc[valid_idx]
        y_fold_train=y_train.iloc[train_idx]
        y_fold_valid=y_train.iloc[valid_idx]

        random_forest_tune.fit(X_fold_train,y_fold_train.values.ravel())

        y_fold_pred=random_forest_tune.predict(X_fold_valid)

        score=f1_score(y_fold_valid,y_fold_pred)

        cv_f1_score.append(score)

    return np.mean(cv_f1_score)
    


In [323]:
study=optuna.create_study(direction='maximize')

study.optimize(objective,n_trials=50)

[I 2026-06-29 23:12:30,113] A new study created in memory with name: no-name-8907cd24-8b5f-4381-b364-8a5ac5457aea
[I 2026-06-29 23:13:41,188] Trial 0 finished with value: 0.7623420511590957 and parameters: {'n_estimators': 450, 'max_depth': 7, 'min_samples_split': 19, 'min_samples_leaf': 3, 'max_features': 'log2'}. Best is trial 0 with value: 0.7623420511590957.
[I 2026-06-29 23:19:00,565] Trial 1 finished with value: 0.8087830551270929 and parameters: {'n_estimators': 400, 'max_depth': 13, 'min_samples_split': 15, 'min_samples_leaf': 1, 'max_features': None}. Best is trial 1 with value: 0.8087830551270929.
[I 2026-06-29 23:19:29,755] Trial 2 finished with value: 0.7576383349136389 and parameters: {'n_estimators': 250, 'max_depth': 5, 'min_samples_split': 16, 'min_samples_leaf': 1, 'max_features': 'sqrt'}. Best is trial 1 with value: 0.8087830551270929.
[I 2026-06-29 23:19:37,871] Trial 3 finished with value: 0.7696049174421407 and parameters: {'n_estimators': 50, 'max_depth': 9, 'min_

In [324]:
study.best_params

{'n_estimators': 450,
 'max_depth': 19,
 'min_samples_split': 2,
 'min_samples_leaf': 5,
 'max_features': None}

In [325]:
study.best_value

0.8490587304728946

In [326]:
study.best_trial

FrozenTrial(number=43, state=<TrialState.COMPLETE: 1>, values=[0.8490587304728946], datetime_start=datetime.datetime(2026, 6, 30, 0, 26, 13, 646508), datetime_complete=datetime.datetime(2026, 6, 30, 0, 28, 56, 486529), params={'n_estimators': 450, 'max_depth': 19, 'min_samples_split': 2, 'min_samples_leaf': 5, 'max_features': None}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=50, step=50), 'max_depth': IntDistribution(high=19, log=False, low=5, step=2), 'min_samples_split': IntDistribution(high=20, log=False, low=2, step=1), 'min_samples_leaf': IntDistribution(high=10, log=False, low=1, step=1), 'max_features': CategoricalDistribution(choices=('sqrt', 'log2', None))}, trial_id=43, value=None)

In [327]:
def objective(trial):
    n_estimators=trial.suggest_int('n_estimators',50,500,step=50)
    max_depth=trial.suggest_int('max_depth',5,19,step=2)
    min_samples_split=trial.suggest_int('min_samples_split',2,20,step=1)
    min_samples_leaf=trial.suggest_int('min_samples_leaf',1,10,step=1)
    max_features=trial.suggest_categorical('max_features',['sqrt','log2',None])
    
    random_forest_tune=RandomForestClassifier(n_estimators=n_estimators,max_depth=max_depth,
                                              min_samples_split=min_samples_split,min_samples_leaf=min_samples_leaf,
                                              max_features=max_features,class_weight='balanced',
                                              bootstrap=True,criterion='gini',random_state=42,n_jobs=-1)
    
    skf = StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
    cv_scores=cross_val_score(estimator=random_forest_tune,X=X_train_processed,y=y_train.values.ravel(),cv=skf,scoring='f1')
    return cv_scores.mean()

In [328]:
study1=optuna.create_study(direction='maximize')

study1.optimize(objective,n_trials=50)

[I 2026-06-30 00:41:58,661] A new study created in memory with name: no-name-73d7c268-50e2-4749-b146-eaf3d2e28799
[I 2026-06-30 00:44:12,360] Trial 0 finished with value: 0.7910598247553062 and parameters: {'n_estimators': 450, 'max_depth': 11, 'min_samples_split': 18, 'min_samples_leaf': 6, 'max_features': None}. Best is trial 0 with value: 0.7910598247553062.
[I 2026-06-30 00:44:47,416] Trial 1 finished with value: 0.8402004656646158 and parameters: {'n_estimators': 100, 'max_depth': 17, 'min_samples_split': 13, 'min_samples_leaf': 1, 'max_features': None}. Best is trial 1 with value: 0.8402004656646158.
[I 2026-06-30 00:45:07,739] Trial 2 finished with value: 0.7831409026102689 and parameters: {'n_estimators': 200, 'max_depth': 11, 'min_samples_split': 19, 'min_samples_leaf': 4, 'max_features': 'sqrt'}. Best is trial 1 with value: 0.8402004656646158.
[I 2026-06-30 00:45:13,123] Trial 3 finished with value: 0.7816880397448079 and parameters: {'n_estimators': 50, 'max_depth': 11, 'min

In [329]:
study1.best_params

{'n_estimators': 100,
 'max_depth': 19,
 'min_samples_split': 5,
 'min_samples_leaf': 1,
 'max_features': None}

In [330]:
study1.best_value

0.8596756563304714

In [331]:
study1.best_trial

FrozenTrial(number=41, state=<TrialState.COMPLETE: 1>, values=[0.8596756563304714], datetime_start=datetime.datetime(2026, 6, 30, 1, 3, 28, 851257), datetime_complete=datetime.datetime(2026, 6, 30, 1, 4, 7, 20083), params={'n_estimators': 100, 'max_depth': 19, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': None}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=50, step=50), 'max_depth': IntDistribution(high=19, log=False, low=5, step=2), 'min_samples_split': IntDistribution(high=20, log=False, low=2, step=1), 'min_samples_leaf': IntDistribution(high=10, log=False, low=1, step=1), 'max_features': CategoricalDistribution(choices=('sqrt', 'log2', None))}, trial_id=41, value=None)

In [332]:
final_random_forest=RandomForestClassifier(**study1.best_params,bootstrap=True,class_weight='balanced',criterion='gini',random_state=42,n_jobs=-1)


In [334]:
final_random_forest.fit(X_train_processed,y_train.values.ravel())

,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",19
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",5
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",None
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"class_weight class_weight: {""balanced"", ""balanced_subsample""}, dict or list of dicts, default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one. Formulti-output problems, a list of dicts can be provided in the sameorder as the columns of y.Note that for multioutput (including multilabel) weights should bedefined for each class of every column in its own dict. For example,for four-class multilabel classification weights should be[{0: 1, 1: 1}, {0: 1, 1: 5}, {0: 1, 1: 1}, {0: 1, 1: 1}] instead of[{1:1}, {2:5}, {3:1}, {4:1}].The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``The ""balanced_subsample"" mode is the same as ""balanced"" except thatweights are computed based on the bootstrap sample for every treegrown.For multi-output, the weights of each column of y will be multiplied.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified.",'balanced'
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches.

In [335]:
final_y_pred_rf=final_random_forest.predict(X_test_processed)

In [337]:
final_y_proba_rf=final_random_forest.predict_proba(X_test_processed)[:,1]

In [339]:
final_metrics_rf=evaluate_model('Random Forest',y_true=y_test.values.ravel(),y_pred=final_y_pred_rf,y_prob=final_y_proba_rf)
final_metrics_rf

Model          : Random Forest
Accuracy       : 0.7275517764527396
Precision      : 0.9339927891282241
Recall         : 0.7624905660377358
F1 Score       : 0.8395728591016745
ROC-AUC        : 0.5859722458256125
Test Samples   : 28343

Confusion Matrix
[[  415  1428]
 [ 6294 20206]]

Classification Report


,precision,recall,f1-score,support
False,0.061857,0.225176,0.097053,1843.000000
True,0.933993,0.762491,0.839573,26500.000000
accuracy,0.727552,0.727552,0.727552,0.727552
macro avg,0.497925,0.493833,0.468313,28343.000000
weighted avg,0.877282,0.727552,0.791291,28343.000000
